In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# IBM Circuit-Funktion

> **Note:** * Qiskit Functions sind ein experimentelles Feature, das ausschließlich für Nutzer des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über die IBM Quantum Platform API) Plan verfügbar ist. Sie befinden sich im Preview-Status und können sich noch ändern.

## Überblick
Die IBM&reg; Circuit function nimmt [abstrakte PUBs](./primitive-input-output) als Eingabe und gibt geminderte Erwartungswerte als Ausgabe zurück. Diese Circuit function umfasst eine automatisierte und individuell angepasste Pipeline, damit sich Forschende auf die Entdeckung neuer Algorithmen und Anwendungen konzentrieren können.

## Beschreibung
Nachdem du deinen PUB eingereicht hast, werden deine abstrakten Circuits und Observablen automatisch transpiliert, auf Hardware ausgeführt und nachbearbeitet, um geminderte Erwartungswerte zurückzugeben. Dafür werden folgende Tools kombiniert:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), einschließlich automatischer Auswahl von KI-gestützten und heuristischen Transpilierungspässen, um abstrakte Circuits in hardware-optimierte ISA-Circuits umzuwandeln
- [Fehlerunterdrückung und -minderung für Berechnungen im Utility-Maßstab](./error-mitigation-and-suppression-techniques), einschließlich Messung und Gate-Twirling, Dynamical Decoupling, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE) und Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), zur Ausführung von ISA-PUBs auf Hardware und Rückgabe geminderter Erwartungswerte

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Erste Schritte
Authentifiziere dich mit deinem [API-Schlüssel](http://quantum.cloud.ibm.com/) und wähle die Qiskit Function wie folgt aus. (Dieser Codeausschnitt setzt voraus, dass du dein Konto bereits [in deiner lokalen Umgebung gespeichert hast](/guides/functions#install-qiskit-functions-catalog-client).)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Beispiel
Probiere zunächst dieses einfache Beispiel aus:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Überprüfe den [Status](/guides/functions#check-job-status) deines Qiskit Function-Workloads oder rufe die [Ergebnisse](/guides/functions#retrieve-results) wie folgt ab:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Die Ergebnisse haben dasselbe Format wie ein [Estimator-Ergebnis](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Eingaben
Die folgende Tabelle zeigt alle Eingabeparameter, die die IBM Circuit function akzeptiert. Der nachfolgende Abschnitt [Optionen](#options) geht detaillierter auf die verfügbaren `options` ein.

| Name      | Typ                        | Beschreibung                                                                                                                                                                                                                        | Erforderlich | Standard                                                                 | Beispiel                                 |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Name des Backends, auf dem die Anfrage ausgeführt wird.                                                                                                                                                                         | Ja       | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Ein iterierbares Objekt aus abstrakten PUB-ähnlichen (Primitive Unified Bloc) Objekten, etwa Tupel `(circuit, observables)` oder `(circuit, observables, parameter_values)`. Siehe [Überblick über PUBs](/guides/primitive-input-output#overview-of-pubs) für weitere Informationen. Die Circuits können abstrakt (nicht-ISA) sein. | Ja       | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Eingabeoptionen. Weitere Details findest du im Abschnitt **Optionen**.                                                                                                                                                              | Nein     | Siehe den Abschnitt **Optionen** für Details.                            | `{"optimization_level": 3}`               |
| instance  | str                        | Der Cloud-Ressourcenname der zu verwendenden Instanz im entsprechenden Format.                                                                                                                                                     | Nein     | Eine Instanz wird zufällig ausgewählt, wenn dein Konto Zugriff auf mehrere Instanzen hat. | `CRN`                   |

### Optionen
#### Struktur
Ähnlich wie bei Qiskit Runtime-Primitiven können Optionen für die IBM Circuit function als verschachteltes Dictionary angegeben werden. Häufig verwendete Optionen wie ``optimization_level`` und ``mitigation_level`` befinden sich auf der ersten Ebene. Weitere, fortgeschrittenere Optionen sind in verschiedene Kategorien gruppiert, z. B. ``resilience``.

#### Standardwerte
Wenn du keinen Wert für eine Option angibst, wird der vom Service festgelegte Standardwert verwendet.

#### Minderungsstufe
Die IBM Circuit function unterstützt außerdem `mitigation_level`. Die Minderungsstufe legt fest, wie viel Fehlerunterdrückung und -minderung auf den Job angewendet wird. Höhere Stufen liefern genauere Ergebnisse, auf Kosten längerer Verarbeitungszeiten. Das Ausmaß der Fehlerreduktion hängt von der angewendeten Methode ab. Die Minderungsstufe abstrahiert die detaillierte Auswahl von Fehlerminderungs- und -unterdrückungsmethoden, damit Nutzer das für ihre Anwendung passende Kosten-/Genauigkeitsverhältnis abwägen können. Die folgende Tabelle zeigt die entsprechenden Methoden für jede Stufe.

> **Note:** Obwohl die Namen ähnlich sind, wendet `mitigation_level` andere Techniken an als die vom `resilience_level` des Estimators verwendeten.

Ähnlich wie ``resilience_level`` im Qiskit Runtime Estimator legt ``mitigation_level`` einen Basissatz kuratierter Optionen fest. Alle Optionen, die du zusätzlich zur Minderungsstufe manuell angibst, werden auf den durch die Minderungsstufe definierten Basissatz angewendet. Grundsätzlich könntest du also die Minderungsstufe auf 1 setzen, aber die Messungsminderung deaktivieren – auch wenn dies nicht empfohlen wird.

| **Minderungsstufe** | **Technik** |
|:-:|:-:|
| 1 [Standard] | Dynamical Decoupling + Messung-Twirling + TREX  |
| 2 | Stufe 1 + Gate-Twirling + ZNE via Gate-Faltung |
| 3 | Stufe 1 + Gate-Twirling + ZNE via PEA |

Das folgende Beispiel zeigt, wie die Minderungsstufe gesetzt wird:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Alle verfügbaren Optionen
Zusätzlich zu `mitigation_level` bietet die IBM Circuit function eine Auswahl an erweiterten Optionen, mit denen du das Kosten-/Genauigkeitsverhältnis fein abstimmen kannst. Die folgende Tabelle zeigt alle verfügbaren Optionen:

| Option | Sub-Option | Sub-Sub-Option | Beschreibung | Auswahlmöglichkeiten | Standard |
|-|-|-|-|-|-|
| default_precision |  |  | Die Standard-Präzision für jeden PUB oder `run()`-Aufruf,<br/>der keine eigene Präzision angibt.<br/>Jeder Eingabe-PUB kann seine eigene Präzision festlegen. Wenn der `run()`-Methode eine Präzision übergeben wird, gilt dieser Wert für alle PUBs im `run()`-Aufruf, die keine eigene Präzision angeben. | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximale Ausführungszeit in Sekunden, basierend auf der<br/>QPU-Nutzung (nicht auf der Wanduhrzeit). QPU-Nutzung ist die<br/>Zeitspanne, in der der QPU ausschließlich für die Verarbeitung deines Jobs genutzt wird. Überschreitet ein Job dieses Zeitlimit, wird er zwangsweise abgebrochen. | Ganzzahl in Sekunden im Bereich [1, 10800] |  |
| mitigation_level |  |  | Wie viel Fehlerunterdrückung und -minderung angewendet wird. Weitere Informationen zu den auf jeder Stufe verwendeten Methoden findest du im Abschnitt [Minderungsstufe](#mitigation-level). | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Wie viel Optimierung an den Circuits durchgeführt wird. [Höhere Stufen](/guides/set-optimization) erzeugen stärker optimierte Circuits, auf Kosten längerer Transpilierungszeit. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Ob Dynamical Decoupling aktiviert werden soll. Eine Erklärung der Methode findest du unter [Fehlerunterdrückung und -minderungstechniken](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling). | True/False | True |
|  | sequence_type |  | Welche Dynamical-Decoupling-Sequenz verwendet werden soll.<br/>* `XX`: verwendet die Sequenz `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: verwendet die Sequenz `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: verwendet die Sequenz<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Ob 2-Qubit-Clifford-Gate-Twirling angewendet werden soll. | True/False | False |
|  | enable_measure |  | Ob Twirling bei Messungen aktiviert werden soll. | True/False | True |
| resilience | measure_mitigation |  | Ob die TREX-Messfehlerminderungsmethode aktiviert werden soll. Eine Erklärung der Methode findest du unter [Fehlerunterdrückung und -minderungstechniken](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex). | True/False | True |
|  | zne_mitigation |  | Ob die Zero-Noise-Extrapolation-Fehlerminderungsmethode aktiviert werden soll. Eine Erklärung der Methode findest du unter [Fehlerunterdrückung und -minderungstechniken](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne). | True/False | False |
|  | zne | amplifier | Welche Technik zur Rauschverstärkung verwendet werden soll. Eine der folgenden: <br/> - `gate_folding` (Standard) verwendet 2-Qubit-Gate-Faltung zur Rauschverstärkung. Wenn der Rauschfaktor nur eine Teilmenge der Gates verstärken muss, werden diese zufällig ausgewählt.<br/><br/> - `gate_folding_front` verwendet 2-Qubit-Gate-Faltung zur Rauschverstärkung. Wenn der Rauschfaktor nur eine Teilmenge der Gates verstärken muss, werden diese vom Anfang des topologisch geordneten DAG-Circuits ausgewählt.<br/><br/> - `gate_folding_back` verwendet 2-Qubit-Gate-Faltung zur Rauschverstärkung. Wenn der Rauschfaktor nur eine Teilmenge der Gates verstärken muss, werden diese vom Ende des topologisch geordneten DAG-Circuits ausgewählt.<br/><br/> - `pea` verwendet eine Technik namens Probabilistic Error Amplification (PEA) zur Rauschverstärkung. Eine Erklärung der Methode findest du unter [Fehlerunterdrückung und -minderungstechniken](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea). | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Rauschfaktoren für die Rauschverstärkung. | Liste von Floats; jeder Float >= 1 | (1, 1.5, 2) für PEA, und (1, 3, 5) andernfalls. |
|  |  | extrapolator | Rauschfaktoren, bei denen die Extrapolationsmodelle ausgewertet werden sollen. Diese Option hat keinen Einfluss auf die Ausführung oder das Modell-Fitting; sie bestimmt lediglich die Punkte, an denen die `extrapolator`-Objekte ausgewertet werden, um in den Datenfeldern `evs_extrapolated` und `stds_extrapolated` zurückgegeben zu werden. | eines oder mehrere von `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Ob die Probabilistic-Error-Cancellation-Fehlerminderungsmethode aktiviert werden soll. Eine Erklärung der Methode findest du unter [Fehlerunterdrückung und -minderungstechniken](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec). | True/False | False |
|  | pec | max_overhead | Der maximal zulässige Circuit-Sampling-Overhead, oder `None` für kein Maximum. | None / Ganzzahl > 1 | 100 |

Im folgenden Beispiel deaktiviert das Setzen der Minderungsstufe auf 1 zunächst die ZNE-Minderung, aber das Setzen von `zne_mitigation` auf `True` überschreibt die entsprechende Einstellung aus `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Ausgaben
Die Ausgabe einer Circuit function ist ein [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), das zwei Felder enthält:

- Ein oder mehrere [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult)-Objekte. Diese können direkt aus dem `PrimitiveResult` indiziert werden.

- Metadaten auf Job-Ebene.

Jedes `PubResult` enthält ein `data`- und ein `metadata`-Feld.

- Das `data`-Feld enthält mindestens ein Array von Erwartungswerten (`PubResult.data.evs`) und ein Array von Standardfehlern (`PubResult.data.stds`). Je nach verwendeten Optionen kann es weitere Daten enthalten.

- Das `metadata`-Feld enthält PUB-Metadaten (`PubResult.metadata`).

Der folgende Codeausschnitt beschreibt das Format von `PrimitiveResult` (und dem zugehörigen `PubResult`).